# Семинар 11. Соревнование «Камни с неба»

По ста измерениям камня оценить, метеорит ли он. Метрика — ROC AUC по всем камням `test` сразу.
Условие, данные и лидерборд — на странице соревнования.

Идём по блокам сверху вниз. Готовые ячейки запускаем, в пустых пишем своё. Каждую идею сначала
оцениваем на своей проверке (блок 1), на Kaggle отправляем лучшее.

In [ ]:
try:
    import catboost
except ImportError:          # в Colab CatBoost не установлен
    %pip install -q catboost

import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from catboost import CatBoostClassifier

## 0. Данные

In [ ]:
# на Kaggle данные соревнования подключаются сами (/kaggle/input/...),
# в Colab и локально — скачиваем с GitHub курса (уже скачанные файлы wget не трогает)
found = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if found:
    DATA = os.path.dirname(found[0])
else:
    DATA = '.'
    URL = 'https://raw.githubusercontent.com/paiflare/ml_hse_2026/master/week11'
    !wget -q -nc {URL}/train.csv {URL}/test.csv

train = pd.read_csv(os.path.join(DATA, 'train.csv'))
test = pd.read_csv(os.path.join(DATA, 'test.csv'))
meas = [c for c in train.columns if c.startswith('m')]   # измерения m00 ... m99

print(train.shape, test.shape)
train.head()

## 1. Своя проверка

Посылок 10 в день, а публичная часть лидерборда (public) — только половина `test`. Поэтому идеи
сравниваем на отложенной выборке (`holdout`): четверти `train`, которую модель не видит при обучении.
Разбиение у всех одинаковое: числа можно сравнивать с соседом и со слайдами.

In [ ]:
tr, va = train_test_split(train, test_size=0.25, random_state=42, stratify=train['site'])
print(tr.shape, va.shape)

scores = {}   # журнал: идея → AUC на проверке

def check(name, pred):
    """pred — оценка для каждой строки va, в том же порядке."""
    assert len(pred) == len(va), 'нужна оценка для каждой строки va'
    scores[name] = roc_auc_score(va['target'], pred)
    print(f'{name}: AUC = {scores[name]:.4f}')

def save_submission(pred_test, path='submission.csv'):
    """pred_test — оценка для каждой строки test, в том же порядке."""
    assert len(pred_test) == len(test), 'нужна оценка для каждой строки test'
    sub = pd.DataFrame({'id': test['id'], 'target': pred_test})
    sub.to_csv(path, index=False)
    print('сохранено:', path, sub.shape)

## 2. Бейзлайн

CatBoost из `baseline.ipynb`: все столбцы как есть. Только облегчённый, чтобы считался быстрее:
500 деревьев глубины 4 вместо 1000 глубины 6. Учим на `tr`, проверяем на `va`.

In [ ]:
features = [c for c in train.columns if c not in ('id', 'target')]   # site + m00...m99

model = CatBoostClassifier(iterations=500, depth=4, learning_rate=0.2, random_seed=0, verbose=0)
model.fit(tr[features], tr['target'])
check('бейзлайн CatBoost', model.predict_proba(va[features])[:, 1])

Первая посылка: та же модель на всём `train`. Отправьте `submission.csv` на Kaggle. Строка
`baseline_submission.csv` на лидерборде чуть выше — там полный CatBoost из `baseline.ipynb`.

In [ ]:
model_full = CatBoostClassifier(iterations=500, depth=4, learning_rate=0.2, random_seed=0, verbose=0)
model_full.fit(train[features], train['target'])
save_submission(model_full.predict_proba(test[features])[:, 1])

## 3. Первый взгляд на данные

- Сколько строк и столбцов, есть ли пропуски, какой баланс классов?
- Что за столбец `site`: сколько значений, сколько камней на участке в `train` и `test`?
- В каких масштабах измерения `m00…m99`? Как выглядят их распределения?
- Есть ли измерение, которое само по себе связано с `target`?

<details><summary>Подсказка</summary>

`isna()`, `value_counts()`, `describe()`, `hist()`, `corrwith()`

</details>

In [ ]:
# ваш код

**Вывод:** 

## 4. Участки

`site` — номер участка от 0 до 31, бейзлайн подаёт его как обычное число.

- Насколько важен `site` для бейзлайна? Дерево может разделить камни условием «`site` < 15.5» —
  есть ли у такого условия смысл, если `site` — просто номер?
- Отличаются ли участки долей метеоритов?

<details><summary>Подсказка</summary>

Важность признаков: `model.get_feature_importance()`. Сравнить участки: `train.groupby('site')['target'].mean()`.

</details>

<details><summary>Ещё подсказка</summary>

В `test` те же 32 участка, что и в `train`. Что, если учить отдельную модель на каждый?

</details>

<details><summary>Каркас цикла по участкам</summary>

```python
pred = np.zeros(len(va))
for s in range(32):
    tr_s = tr[tr['site'] == s]            # обучение: строки участка s
    in_s = (va['site'] == s).values       # где в va строки участка s
    # начните с логрег: make_pipeline(StandardScaler(), LogisticRegression());
    # CatBoost × 32 участка на всех столбцах считается минуты
    # model = ...
    # model.fit(...)
    # pred[in_s] = model.predict_proba(...)[:, 1]
check('...', pred)
```

</details>

In [ ]:
# ваш код

**Вывод:** 

## 5. Какие измерения нужны участку

Цикл по участкам из блока 4 — готовой функцией, дальше она понадобится в каждом блоке.

In [ ]:
def by_site(model, cols, fit_df=tr, pred_df=va):
    """Учит копию model отдельно на каждом участке fit_df, предсказывает строки pred_df.
    cols — список столбцов или словарь {участок: список столбцов}.
    Возвращает оценки в порядке строк pred_df."""
    pred = np.zeros(len(pred_df))
    for s in range(32):
        c = cols[s] if isinstance(cols, dict) else cols
        fit_s = fit_df[fit_df['site'] == s]
        in_s = (pred_df['site'] == s).values
        m = clone(model)                       # новая необученная копия модели
        m.fit(fit_s[c], fit_s['target'])
        pred[in_s] = m.predict_proba(pred_df.loc[in_s, c])[:, 1]
    return pred

# пример: check('логрег по участкам', by_site(make_pipeline(StandardScaler(), LogisticRegression()), meas))

На участке 384 камня в `tr` и 100 измерений.

- Все ли измерения нужны каждому участку?
- Чем полезное измерение может отличаться от шума, если смысл столбцов неизвестен?
- Отберите измерения для каждого участка и повторите логрег по участкам только на них. Сколько дал
  отбор?

<details><summary>Подсказка</summary>

Сравните распределение одного и того же измерения на разных участках: среднее, разброс.

</details>

<details><summary>Ещё подсказка</summary>

Единицы у столбцов разные. Прежде чем сравнивать разброс разных столбцов, приведите их к одному масштабу. Гистограмма по всем парам «участок × измерение» подскажет порог.

</details>

<details><summary>Как собрать таблицу и отбор</summary>

```python
# z = ...                                  # измерения train, приведённые к одному масштабу
sd = z.groupby(train['site']).std()        # таблица: 32 участка × 100 измерений
plt.hist(sd.values.ravel(), bins=100)      # все 3200 значений на одной гистограмме
keep = {s: sd.columns[sd.loc[s] > ...].tolist() for s in range(32)}   # порог — по гистограмме
```

</details>

In [ ]:
# ваш код

**Вывод:** 

## 6. Какая модель лучше

На тех же отобранных измерениях замените логрег из блока 5 на модели, которые уже знаете:
случайный лес, бустинг, kNN (kNN — со `StandardScaler`: он считает расстояния). CatBoost — с
параметрами бейзлайна, тогда 32 участка считаются меньше минуты.

- Какая модель лучше и насколько по сравнению с логрег?
- Если какая-то модель обошла логрег на тех же измерениях — почему?

<details><summary>Подсказка</summary>

Логрег считает $w_1 x_1 + \dots + w_k x_k$. Что умеет дерево и не умеет эта сумма?

</details>

In [ ]:
# ваш код

**Вывод:** 

## 7. Новые признаки

Если логрег проигрывает деревьям, можно не менять модель, а дать ей новые признаки. Придумайте
новые столбцы для логистической регрессии и сравните с лучшей моделью блока 6.

<details><summary>Подсказка</summary>

Из каких признаков дерево собирает правило «$x_1$ большой и $x_2$ большой»? Какой новый столбец мог бы подсказать это логрегу?

</details>

<details><summary>Ещё подсказка</summary>

`sklearn.preprocessing.PolynomialFeatures`:

```python
PolynomialFeatures(2).fit_transform([[2, 3]])   # [[1, 2, 3, 4, 6, 9]]: 1, x1, x2, x1², x1·x2, x2²
```

Признаков станет много: масштабируйте до и после `PolynomialFeatures` и подберите регуляризацию —
параметр `C` в `LogisticRegression`, меньше `C` — сильнее.

</details>

In [ ]:
# ваш код

**Вывод:** 

## 8. Посылка

Лучшую схему учим на всём `train` и предсказываем `test`. AUC считают по всем камням `test` сразу,
поэтому оценки разных участков должны быть сравнимы между собой.

In [ ]:
# лучшую схему — на всём train, оценки для test:
# save_submission(by_site(модель, столбцы, train, test))

In [ ]:
# журнал идей: что сколько дало на проверке
pd.Series(scores).sort_values()